# Σύστημα Β – Βήμα 5: 10 εκτελέσεις σε 4 διαχωρισμούς (90-10, 80-20, 70-30, 60-40)
Ζήτηση του επιβλέποντα: κάθε εκδοχή να αξιολογείται με **10 ανεξάρτητες εκτελέσεις**, σε διαχωρισμούς εκπαίδευσης – ελέγχου
**90/10, 80/20, 70/30 και 60/40**, με τυχαία επιλογή των δειγμάτων κάθε φορά.

Στα B01–B03 χρησιμοποιήσαμε έναν διαχωρισμό 70/15/15 (σπόρος 42). Εδώ το ίδιο δίκτυο (9 είσοδοι, 16 νευρώνες) και τα ίδια μοντέλα σύγκρισης
εκπαιδεύονται ξανά 40 φορές. **Καμία νέα ρύθμιση**: το μέγεθος και το dropout είναι αυτά που επιλέχθηκαν στο B02.

**Κάθε εκτέλεση:**
1. τυχαίος διαχωρισμός εκπαίδευση / έλεγχος (stratify στον στόχο)·
2. από το κομμάτι εκπαίδευσης κρατάμε 15% ως validation, μόνο για το early stopping του δικτύου·
3. προεπεξεργασία που μαθαίνεται **μόνο** από το κομμάτι εκπαίδευσης·
4. νευρωνικό, κανόνας 3 ερωτήσεων (δέντρο βάθους 3), δέντρο βάθους 5 και Gradient Boosting στα ίδια δεδομένα· αξιολόγηση στο κομμάτι ελέγχου.

Η εκτέλεση k χρησιμοποιεί σπόρο k (1–10) και για τον διαχωρισμό και για τα μοντέλα.

**Χρόνος:** περίπου 20–60 λεπτά. Κάθε εκτέλεση αποθηκεύεται μόλις τελειώσει στο `models/intrusion_splits_runs.csv`·
αν το διακόψεις, ξανατρέχεις το notebook και συνεχίζει από εκεί που σταμάτησε.

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
RESULTS = Path(MODELS + "intrusion_splits_runs.csv")
HISTORY = Path(MODELS + "intrusion_splits_history.json")
SPLITS = [0.9, 0.8, 0.7, 0.6]
RUNS = list(range(1, 11))
VAL_FRACTION = 0.15

with open(MODELS + "intrusion_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
with open(MODELS + "intrusion_mlp_config.json", encoding="utf-8") as f:
    cfg = json.load(f)
TARGET = meta["target"]
NUM_COLS, BIN_COLS, CAT_COLS = meta["num_cols"], meta["bin_cols"], meta["cat_cols"]
LOG_COLS = list(meta.get("log_cols", []))
PLAIN_COLS = [c for c in NUM_COLS if c not in LOG_COLS]
FEATURES = meta["features"]
UNITS, DROPOUT = tuple(cfg["units"]), cfg["dropout"]
EPOCHS, PATIENCE, BATCH = cfg.get("epochs_max", 200), cfg.get("patience", 15), cfg.get("batch_size", 128)
print("Είσοδοι:", FEATURES)
print("Δίκτυο:", UNITS, "| dropout:", DROPOUT, "   (αναμενόμενο: (16,) | 0.1)")

df = pd.read_csv(DATA + "intrusion_clean.csv", keep_default_na=False)
raw = pd.read_csv(DATA + "cybersecurity_intrusion_data.csv")
assert len(raw) == len(df) and (raw[TARGET].to_numpy() == df[TARGET].to_numpy()).all(), "Οι γραμμές δεν ταιριάζουν (βλ. B01)"
df["browser_type"] = raw["browser_type"].to_numpy()
Y = df[TARGET].to_numpy("int64")
print("Γραμμές:", len(df), f"| επιθέσεις {Y.mean():.1%}")

## Β5.1 Συναρτήσεις
Ίδια προεπεξεργασία με το B01 και ίδιο δίκτυο με το B02.

In [ ]:
def make_split(train_frac, seed):
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, train_size=train_frac, stratify=Y, random_state=seed)
    tr, va = train_test_split(tr, test_size=VAL_FRACTION, stratify=Y[tr], random_state=seed)
    return {"train": tr, "val": va, "test": te}

def make_preprocessor():
    steps = []
    if LOG_COLS:
        steps.append(("log", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                       ("scale", StandardScaler())]), LOG_COLS))
    if PLAIN_COLS:
        steps.append(("num", StandardScaler(), PLAIN_COLS))
    steps.append(("bin", "passthrough", BIN_COLS))
    steps.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS))
    return ColumnTransformer(steps)

def build_model(n_in):
    inp = keras.Input(shape=(n_in,), name="session_features")
    x = inp
    for i, u in enumerate(UNITS):
        x = layers.Dense(u, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_{i+1}")(x)
        x = layers.Dropout(DROPOUT, name=f"dropout_{i+1}")(x)
    out = layers.Dense(1, activation="sigmoid", name="attack")(x)
    model = keras.Model(inp, out, name="intrusion_mlp_split")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy")
    return model

def metrics(y, p):
    pred = (p >= 0.5).astype(int)
    return {"accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred), "F1": f1_score(y, pred),
            "FPR": ((pred == 1) & (y == 0)).sum() / max((y == 0).sum(), 1),
            "ROC-AUC": roc_auc_score(y, p), "loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1])}

RULE = "Κανόνας (δέντρο βάθους 3)"
def run_once(train_frac, seed):
    ix = make_split(train_frac, seed)
    pre = make_preprocessor()
    X = {"train": pre.fit_transform(df.iloc[ix["train"]][FEATURES]).astype("float32")}
    for s in ("val", "test"):
        X[s] = pre.transform(df.iloc[ix[s]][FEATURES]).astype("float32")
    y = {s: Y[ix[s]] for s in ix}
    common = {"split": f"{round(train_frac * 100)}-{round((1 - train_frac) * 100)}", "run": seed,
              "n_train": len(ix["train"]), "n_val": len(ix["val"]), "n_test": len(ix["test"])}
    rows, preds = [], {}

    t0 = time.time()
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    model = build_model(X["train"].shape[1])
    hist = model.fit(X["train"], y["train"].astype("float32"), validation_data=(X["val"], y["val"].astype("float32")),
                     epochs=EPOCHS, batch_size=BATCH, verbose=0,
                     callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
                                keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)])
    p = np.asarray(model.predict(X["test"], batch_size=1024, verbose=0), dtype="float64").ravel()
    preds["Νευρωνικό"] = p
    rows.append({"model": "Νευρωνικό", **common, **metrics(y["test"], p),
                 "epochs": len(hist.history["loss"]), "seconds": time.time() - t0})
    history = {k: [float(v) for v in hist.history.get(k, [])] for k in ("loss", "val_loss")}

    others = {RULE: DecisionTreeClassifier(max_depth=3, random_state=seed),
              "Δέντρο βάθους 5": DecisionTreeClassifier(max_depth=5, random_state=seed),
              "Gradient Boosting": GradientBoostingClassifier(random_state=seed)}
    for name, m in others.items():
        t0 = time.time()
        p = m.fit(X["train"], y["train"]).predict_proba(X["test"])[:, 1]
        preds[name] = p
        rows.append({"model": name, **common, **metrics(y["test"], p), "seconds": time.time() - t0})

    same = ((preds["Νευρωνικό"] >= 0.5) == (preds[RULE] >= 0.5)).mean()
    rows[0]["ίδια απόφαση με κανόνα"] = same
    return rows, history

## Β5.2 Οι 40 εκτελέσεις

In [ ]:
done = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame(columns=["model", "split", "run"])
histories = json.loads(HISTORY.read_text(encoding="utf-8")) if HISTORY.exists() else {}
start = time.time()
for train_frac in SPLITS:
    split_name = f"{round(train_frac * 100)}-{round((1 - train_frac) * 100)}"
    for seed in RUNS:
        if ((done["split"] == split_name) & (done["run"] == seed) & (done["model"] == "Νευρωνικό")).any():
            continue
        rows, history = run_once(train_frac, seed)
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_csv(RESULTS, index=False)
        histories[f"{split_name}/{seed}"] = history
        HISTORY.write_text(json.dumps(histories), encoding="utf-8")
        r = {row["model"]: row for row in rows}
        print(f"{split_name} εκτέλεση {seed:2d} | ΝΔ acc {r['Νευρωνικό']['accuracy']:.4f} | κανόνας {r[RULE]['accuracy']:.4f} "
              f"| δέντρο 5 {r['Δέντρο βάθους 5']['accuracy']:.4f} | GBM {r['Gradient Boosting']['accuracy']:.4f} "
              f"| ίδια με κανόνα {r['Νευρωνικό']['ίδια απόφαση με κανόνα']:.1%} | {(time.time() - start) / 60:.0f} λεπτά", flush=True)
print("Ολοκληρώθηκαν:", len(done[done["model"] == "Νευρωνικό"]), "/", len(SPLITS) * len(RUNS))

## Β5.3 Πίνακες ανά μετρική (όπως στο πρότυπο): 10 εκτελέσεις και μέσος όρος – νευρωνικό
Αποθηκεύονται και σε `models/intrusion_splits_table_<μετρική>.csv`.

In [ ]:
res = pd.read_csv(RESULTS)
split_order = [f"{round(s * 100)}-{round((1 - s) * 100)}" for s in SPLITS]
nn = res[res["model"] == "Νευρωνικό"]
pd.set_option("display.width", 200)

def per_run_table(d, metric):
    t = d.pivot(index="run", columns="split", values=metric)[split_order]
    t.loc["Μέσος όρος"] = t.mean()
    t.loc["Τυπ. απόκλιση"] = t.iloc[:-1].std()
    return t

for metric in ["accuracy", "precision", "recall", "F1", "loss", "ROC-AUC", "FPR"]:
    t = per_run_table(nn, metric)
    t.to_csv(MODELS + f"intrusion_splits_table_{metric}.csv")
    print(f"\n=== Νευρωνικό – {metric} ===")
    print(t.round(4).to_string())

## Β5.4 Σύνοψη: όλα τα μοντέλα, μέσος όρος ± τυπική απόκλιση

In [ ]:
cols = ["accuracy", "precision", "recall", "F1", "FPR", "ROC-AUC"]
model_order = ["Νευρωνικό", RULE, "Δέντρο βάθους 5", "Gradient Boosting"]
summary = []
for model in model_order:
    for s in split_order:
        g = res[(res["model"] == model) & (res["split"] == s)]
        summary.append({"μοντέλο": model, "διαχωρισμός": s, **{c: f"{g[c].mean():.4f} ± {g[c].std():.4f}" for c in cols}})
summary = pd.DataFrame(summary)
summary.to_csv(MODELS + "intrusion_splits_summary.csv", index=False)
print(summary.to_string(index=False))

print("\nΊδια απόφαση νευρωνικού και κανόνα (μέσος όρος):",
      nn.groupby("split")["ίδια απόφαση με κανόνα"].mean().reindex(split_order).map("{:.1%}".format).to_dict())
print("Διαχωρισμός 70/15/15 των B01–B03 (test): νευρωνικό acc 0.8826, recall 0.7422, FPR 0.0038 · κανόνας 0.8630")

pair = res.pivot_table(index=["split", "run"], columns="model", values="accuracy")
for other in model_order[1:]:
    wins = (pair["Νευρωνικό"] > pair[other]).groupby(level=0).sum()
    print(f"Νευρωνικό > {other} (ακρίβεια):", {s: f"{int(wins.get(s, 0))}/10" for s in split_order})

## Β5.5 Γραφήματα
- **B06:** κατανομή των 10 εκτελέσεων ανά διαχωρισμό, για όλα τα μοντέλα.
- **B07:** καμπύλες εκπαίδευσης του νευρωνικού (εκτέλεση 1) για κάθε διαχωρισμό.

In [ ]:
COL = {"Νευρωνικό": "#2a78d6", RULE: "#8a8a85", "Δέντρο βάθους 5": "#1baf7a", "Gradient Boosting": "#eb6834"}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, (metric, title) in zip(axes, [("accuracy", "Ακρίβεια"), ("ROC-AUC", "ROC-AUC")]):
    pos = np.arange(len(split_order))
    w = 0.8 / len(model_order)
    for j, model in enumerate(model_order):
        data = [res[(res["model"] == model) & (res["split"] == s)][metric].to_numpy() for s in split_order]
        bp = ax.boxplot(data, positions=pos + (j - (len(model_order) - 1) / 2) * w, widths=w * 0.85, patch_artist=True,
                        medianprops={"color": "black"})
        for box in bp["boxes"]:
            box.set(facecolor=COL[model], alpha=0.85)
    ax.set_xticks(pos, split_order)
    ax.set_xlabel("Διαχωρισμός εκπαίδευσης – ελέγχου (%)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=COL[m], alpha=0.85) for m in model_order]
fig.legend(handles, model_order, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Σύστημα Β: 10 εκτελέσεις ανά διαχωρισμό (σύνολο ελέγχου κάθε εκτέλεσης)")
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(FIG + "B06_splits_10runs_intrusion.png", dpi=200, bbox_inches="tight")
plt.show()

histories = json.loads(HISTORY.read_text(encoding="utf-8"))
fig, axes = plt.subplots(1, len(split_order), figsize=(16, 3.6), sharey=True)
for ax, s in zip(axes, split_order):
    h = histories.get(f"{s}/1", {})
    ep = np.arange(1, len(h.get("loss", [])) + 1)
    ax.plot(ep, h.get("loss", []), color="#2a78d6", lw=2, label="εκπαίδευση")
    ax.plot(ep, h.get("val_loss", []), color="#eb6834", lw=2, label="validation")
    ax.set_title(f"Διαχωρισμός {s}")
    ax.set_xlabel("Εποχή")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Σφάλμα (binary cross-entropy)")
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG + "B07_splits_curves_intrusion.png", dpi=200, bbox_inches="tight")
plt.show()
print("Αποθηκεύτηκαν: figures/B06_splits_10runs_intrusion.png, figures/B07_splits_curves_intrusion.png")